# W2.2 — Split LP cố định

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** Cora; data_protocol.json (W2.1 tải cache).

Chia trên unique canonical pairs: n_val=floor(0.05M), n_test=floor(0.10M), train lấy phần dư.
Negative pool loại toàn bộ positives kể cả held-out. Tỷ lệ 1:1 sau canonicalization.
Full graph chỉ là tập loại trừ; message passing chỉ dùng train positives hai chiều.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
import numpy as np
import torch
from utils.experiment import (
    read_json,
    write_json,
    seed_everything,
    array_hash,
    mapping_hash,
    environment,
    new_run,
    publish_result,
    expect_error,
)
from utils.link_graph import (
    validate_edges,
    canonicalize,
    bidirectional,
    with_self_loops,
    neighborhoods,
    pair_set,
    load_cora,
    make_split,
    audit_split,
)


D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
print(config)

{'dataset': 'Cora', 'public_node_split': 'public', 'seed': 42, 'undirected': True, 'val_ratio': 0.05, 'test_ratio': 0.1, 'negative_ratio': 1, 'split_id': 'cora_lp_v1', 'self_loops': 'exclude_from_pairs_add_once_per_node_in_layer', 'rounding': 'floor_val_floor_test_train_remainder'}


In [ ]:
data_config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
data = load_cora(PROJECT_ROOT,data_config)
full_pairs = canonicalize(data.edge_index.numpy(),data.num_nodes)
fingerprints = {'features':array_hash(data.x.numpy()),'full_pairs':array_hash(full_pairs)}

In [ ]:
before = full_pairs.copy()
arrays = make_split(full_pairs,data.num_nodes,config)
assert np.array_equal(before,full_pairs)
checks = audit_split(arrays,full_pairs,data.num_nodes,bidirectional(arrays['train_pos'],data.num_nodes))
run = new_run(PROJECT_ROOT,'w2_split',config)
manifest = {'schema_version':1,'split_id':config['split_id'],'seed':config['seed'],
            'ratios':{'train':.85,'val':config['val_ratio'],'test':config['test_ratio']},
            'rounding':config['rounding'],'negative_ratio':config['negative_ratio'],'num_nodes':data.num_nodes,
            'config':config,'counts':{k:v.shape[1] for k,v in arrays.items()},
            'input_fingerprints':fingerprints,'array_hashes':{k:array_hash(v) for k,v in arrays.items()},
            'split_hash':mapping_hash(arrays),'dataset':{'name':'Cora','loader':'PyG Planetoid public','node_order':'unchanged'},
            'environment':read_json(run/'environment.json'),'creator_notebook':'notebooks/03_link_prediction/01_edge_split.ipynb',
            'run_id':run.name}
split_dir = PROJECT_ROOT/'artifacts/splits'/config['split_id']
split_dir.mkdir(parents=True,exist_ok=True)
if (split_dir/'edges.npz').exists() or (split_dir/'manifest.json').exists():
    old = read_json(split_dir/'manifest.json')
    with np.load(split_dir/'edges.npz',allow_pickle=False) as archive:
        old_arrays = {k:archive[k] for k in archive.files}
    assert old['config'] == config and old['input_fingerprints'] == fingerprints, 'Không ghi đè v1; cần split ID mới'
    assert mapping_hash(old_arrays) == old['split_hash'] == manifest['split_hash'], 'Split cũ khác generator'
    manifest = old
    print('Giữ nguyên split đã có; xác nhận tái lập thành công.')
else:
    np.savez_compressed(split_dir/'edges.npz',**arrays)
    write_json(split_dir/'manifest.json',manifest)
write_json(run/'metrics.json',{'split_hash':manifest['split_hash'],'counts':manifest['counts'],'checks':checks})
print(manifest['counts']); print('split_hash:',manifest['split_hash']); print('NPZ bytes:',(split_dir/'edges.npz').stat().st_size)

Giữ nguyên split đã có; xác nhận tái lập thành công.
{'train_pos': 4488, 'train_neg': 4488, 'val_pos': 263, 'val_neg': 263, 'test_pos': 527, 'test_neg': 527}
split_hash: 9febb7a31cc7f0362d6c4d39bf7c7a37d6af96a3b9f2ea0f8c7b8630e2cb8c98
NPZ bytes: 47850


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Vì sao không split từng cột cạnh có hướng?
2. Vì sao negative phải loại cả held-out positives?
3. Muốn đổi seed phải tạo artifact như thế nào?